<a href="https://colab.research.google.com/github/Likhithaa-Guntaka/ai-skills-market-intelligence/blob/main/notebooks/02_data_collection_and_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Dealing With Data – Phase II**
## Likhitha Guntaka  

### **Project Title:** Tracking the AI Skills Revolution: How Demand for AI & Data Skills Is Changing

**Phase II — data collection and cleaning.** This notebook connects to both APIs, flattens the responses into DataFrames, and prepares them for analysis. It intentionally retrieves a single page of 50 Adzuna postings; pagination to 150 is introduced in Phase III, and the final results are in `04_final_analysis.ipynb`.

*The Adzuna `/search` endpoint returns only currently-listed postings, so this is a recent US snapshot rather than historical coverage back to 2022.*

# d1: Library Imports

In [1]:
# Used to send requests to online APIs
import requests

# Used to store and manipulate data in tables (DataFrames)
import pandas as pd

# Used for numerical operations (like handling missing values)
import numpy as np

# Used to work with dates and timestamps
from datetime import datetime

# d2: Data Pre-Processing

## AP1-1: Adzuna (Jobs API)
**Connecting to Adzuna Jobs API**
* This API provides real job postings related to AI, Data, and Technology.
* The request uses the Adzuna **US** endpoint (`/jobs/us/`), so this is **US job posting data**, not global data.
* A city filter can be switched ON to narrow results to a single US location (for example New York or San Francisco). It is left OFF here, so results cover the US as a whole.
* The `/search` endpoint returns only **currently-listed** postings, so this notebook is a recent snapshot and provides **no historical coverage back to 2022**.


In [2]:
# CITY FILTER SWITCH
# False = US-wide results across the Adzuna US endpoint (default)
# True  = narrow the search to one US city / location
use_city_filter = False

# Any US location, e.g. "New York", "San Francisco", "Chicago"
city_name = "New York"

# Only need to change ONE word to switch


In [3]:
# d2: Connect to Adzuna API

# Credentials are loaded from Colab Secrets (sidebar -> Secrets / key icon).
# Add ADZUNA_APP_ID and ADZUNA_APP_KEY there and enable notebook access.
# Outside Colab, set them as environment variables of the same name.
import os

try:
    from google.colab import userdata
    APP_ID = userdata.get("ADZUNA_APP_ID")
    APP_KEY = userdata.get("ADZUNA_APP_KEY")
except ImportError:
    APP_ID = os.environ["ADZUNA_APP_ID"]
    APP_KEY = os.environ["ADZUNA_APP_KEY"]

# Using US Adzuna endpoint for job market analysis
adzuna_url = "https://api.adzuna.com/v1/api/jobs/us/search/1"

# Parameters for the request
params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": 50,
    "what": "AI OR Data OR Machine Learning"
}

# If the city filter is ON, add location to the query
if use_city_filter:
    params["where"] = city_name

# Header to avoid being blocked
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Send request to Adzuna
response = requests.get(adzuna_url, params=params, headers=headers)

# Confirm request success
# Redact credentials before printing so they never land in saved notebook output
safe_url = response.url.replace(APP_ID, "<APP_ID>").replace(APP_KEY, "<APP_KEY>")
print("Final URL:", safe_url)
print("Status Code:", response.status_code)
print("Using city filter:", use_city_filter)
print("Scope:", city_name if use_city_filter else "US-WIDE")

Final URL: https://api.adzuna.com/v1/api/jobs/us/search/1?app_id=<APP_ID>&app_key=<APP_KEY>&results_per_page=50&what=AI+OR+Data+OR+Machine+Learning
Status Code: 200
Using city filter: False
City: GLOBAL


### Converting Adzuna data into a DataFrame

* Each row will represent one job posting.
* Each column will represent job details (title, salary, skills, etc.).

In [4]:
# Convert to JSON
adzuna_raw = response.json()

# Store jobs here
adzuna_jobs = []

# Loop through job results
for job in adzuna_raw["results"]:
    adzuna_jobs.append({
        "title": job.get("title"),
        "company": job.get("company", {}).get("display_name"),
        "location": job.get("location", {}).get("display_name"),
        "salary_min": job.get("salary_min"),
        "salary_max": job.get("salary_max"),
        "category": job.get("category", {}).get("label"),
        "created": job.get("created"),
        "description": job.get("description")
    })

# Convert to DataFrame
adzuna_df = pd.DataFrame(adzuna_jobs)

# Show size of dataset
print("Rows:", adzuna_df.shape[0])
print("Columns:", adzuna_df.shape[1])

adzuna_df.head(21)

Rows: 50
Columns: 8


,title,company,location,salary_min,salary_max,category,created,description
0,"Senior Director, AI / Machine Learning Data En...",BNY,"New York City, New York",160465.33,160465.33,IT Jobs,2026-09-02T07:52:40Z,hackajob is collaborating with BNY to connect ...
1,Data Scientist - Ai/machine Learning with Secu...,Markon,"Savage, Anne Arundel County",177234.48,177234.48,IT Jobs,2026-07-09T15:08:13Z,"PLEX Solutions, A Part of Markon, is looking f..."
2,"Delivery Consultant- AI/ML, Data & Machine Lea...",Amazon,"Herndon, Fairfax County",178415.83,178415.83,IT Jobs,2026-07-13T20:56:20Z,Description The Amazon Web Services Profession...
3,"Delivery Consultant- AI/ML, Data & Machine Lea...",Amazon,"Fort Myer, Arlington County",178606.33,178606.33,IT Jobs,2026-07-13T20:56:21Z,Description The Amazon Web Services Profession...
4,"Delivery Consultant- AI/ML, Data & Machine Lea...",Amazon,"Fort Myer, Arlington County",167851.06,167851.06,IT Jobs,2026-07-13T20:56:21Z,Description The Amazon Web Services Profession...
5,"Delivery Consultant- AI/ML, Data & Machine Lea...",Amazon,"Herndon, Fairfax County",167544.95,167544.95,IT Jobs,2026-07-13T20:56:20Z,Description The Amazon Web Services Profession...
6,"Delivery Consultant- AI/ML, Data & Machine Lea...",Amazon,"Herndon, Fairfax County",167476.58,167476.58,IT Jobs,2026-07-13T20:56:20Z,Description The Amazon Web Services Profession...
7,"Delivery Consultant- AI/ML, Data & Machine Lea...",Amazon,"Fort Myer, Arlington County",167655.08,167655.08,IT Jobs,2026-07-13T20:56:20Z,Description The Amazon Web Services Profession...
8,"Delivery Consultant- AI/ML, Data & Machine Lea...",Amazon,"Fort Myer, Arlington County",167723.69,167723.69,IT Jobs,2026-07-13T20:56:21Z,Description The Amazon Web Services Profession...
9,"Delivery Consultant- AI/ML, Data & Machine Lea...",Amazon,"Herndon, Fairfax County",167672.34,167672.34,IT Jobs,2026-07-13T20:56:20Z,Description The Amazon Web Services Profession...


### Cleaning and formatting Adzuna data
In this step, Data is prepared the for analysis by:
- Cleaning column names
- Fixing date format
- Handling missing salary values


In [5]:
# Standardize column names
adzuna_df.columns = [col.lower().strip().replace(" ", "_") for col in adzuna_df.columns]

# Convert created date
adzuna_df["created"] = pd.to_datetime(adzuna_df["created"], errors="coerce")

# Handle missing salary values
adzuna_df["salary_min"] = adzuna_df["salary_min"].fillna(0)
adzuna_df["salary_max"] = adzuna_df["salary_max"].fillna(0)

# Check if any nulls remain
adzuna_df.isnull().sum()

,0
title,0
company,1
location,0
salary_min,0
salary_max,0
category,0
created,0
description,0


### Creating an AI keyword indicator

This column checks if the job description contains important AI-related skills.

In [6]:
import re

# Multi-word and unambiguous terms are safe as plain substring matches.
keywords = [
    "artificial intelligence",
    "machine learning",
    "deep learning",
    "data scientist",
    "llm",
    "python",
    "neural network"
]

# "ai" and "ml" are short enough to appear inside unrelated words such as
# "available", "training", "maintain", "email", "html" and "xml", so they are
# matched as standalone terms only. Word boundaries still allow the forms that
# matter: "AI/ML", "AI-powered", "generative AI", "ml-ops", "ML pipelines".
short_terms = re.compile(r"\b(?:ai|ml)\b")

adzuna_df["contains_ai_keywords"] = adzuna_df["description"].str.lower().apply(
    lambda x: (any(word in x for word in keywords) or bool(short_terms.search(x)))
    if isinstance(x, str) else False
)

# Show 10 random rows as sample
adzuna_df.sample(21)[["title", "location", "contains_ai_keywords"]]

,title,location,contains_ai_keywords
27,"EPM, Applied Machine Learning, AI & Data Platf...","Sunnyvale, Santa Clara County",True
31,AI Data Scientist - Gravitas Recruitment Group,"Grand Central, Manhattan",True
48,GEOINT SIGINT Data Scientist,"Chantilly, Fairfax County",True
13,Senior Data Scientist / AI Machine Learning Re...,"Glendale, Denver",True
32,Intern - Predictive Sales Prospecting,US,True
41,Senior Data Scientist with Security Clearance,"State Farm, Arlington County",True
29,"Vice President, Data Science - Aristocrat","Enterprise, Clark County",True
8,"Delivery Consultant- AI/ML, Data & Machine Lea...","Fort Myer, Arlington County",True
47,Portuguese Linguist for AI Training (Brazil),US,True
12,Principal AI Data Scientist – Scientific AI & ...,"Santa Clara, Santa Clara County",True


In [7]:
print("Percentage of jobs mentioning AI-related keywords:",
      (adzuna_df["contains_ai_keywords"].sum() / len(adzuna_df)) * 100, "%")

Percentage of jobs mentioning AI-related keywords: 96.0 %


## API-2: Connecting to StackExchange (Stack Overflow) API
This API provides data on developer activity by showing
the most popular technology and programming tags.


In [8]:
# This is the main URL for the Stack Exchange / Stack Overflow API
# It allows us to access data about developer questions & tags
so_url = "https://api.stackexchange.com/2.3/tags"


# These are the parameters we send to the API
params = {

    # "desc" means we want results in descending (highest → lowest) order
    "order": "desc",

    # "popular" means return the most popular tags based on activity
    "sort": "popular",

    # We specifically want data from the Stack Overflow website
    "site": "stackoverflow",

    # This limits the response to the top 100 tags
    "pagesize": 100
}


# Send a GET request to the API using the URL and parameters
so_response = requests.get(so_url, params=params)


# Print the status code to confirm successful connection
# 200 = Success
print("Status Code:", so_response.status_code)


# Convert the raw API response into JSON format (a Python-readable dictionary)
so_data = so_response.json()


# Print the number of tags that were pulled from the API
print("Number of tags pulled:", len(so_data["items"]))

Status Code: 200
Number of tags pulled: 100


### Converting Stack Overflow data into a DataFrame

In [9]:
# Create an empty list to store the tag data
tags = []


# Loop through each item (each technology / skill tag)
for item in so_data["items"]:

    # Add the required information for each tag into the list
    tags.append({
        "tag_name": item.get("name"),               # Name of the technology / skill
        "question_count": item.get("count"),        # Number of questions related to this tag
        "has_synonyms": item.get("has_synonyms")    # Whether the tag has similar versions
    })


# Convert the list of dictionaries into a pandas DataFrame (table)
stackoverflow_df = pd.DataFrame(tags)


# Print how many rows and columns the DataFrame has
print("Rows:", stackoverflow_df.shape[0])
print("Columns:", stackoverflow_df.shape[1])


# Display the rows of the DataFrame
stackoverflow_df.head(21)

Rows: 100
Columns: 3


,tag_name,question_count,has_synonyms
0,.net,340237,True
1,ajax,220324,True
2,algorithm,121430,True
3,amazon-web-services,158630,True
4,android,1413874,True
5,android-studio,90157,True
6,angular,306395,True
7,angularjs,261236,True
8,apache,91471,True
9,apache-spark,82510,True


### Cleaning Stack Overflow data

In [10]:
# Convert question_count to numeric (just in case it was read as text)
stackoverflow_df["question_count"] = pd.to_numeric(
    stackoverflow_df["question_count"],
    errors="coerce"   # If there are bad values, turn them into NaN
)


# Check if there are any missing (null) values in the dataset
stackoverflow_df.isnull().sum()


# Show the top 10 most popular technology skills
# (The ones with the highest question_count appear first)
stackoverflow_df.sort_values(by="question_count", ascending=False).head(21)

,tag_name,question_count,has_synonyms
43,javascript,2521824,True
67,python,2204680,True
42,java,1914472,True
17,c#,1621768,True
64,php,1461062,True
4,android,1413874,True
38,html,1185146,True
44,jquery,1029947,True
18,c++,814062,True
19,css,805631,True
